# Clase 6: Bonus AutoML y NLP Clásico
---

<img src="../img/sergio_portrait_square.png" alt="Sergio Benito" align="right" width="100">

- **Autor:** Sergio Benito Martín
- **Contacto:** pontia@sergiobenito.com
- **Última actualización:** 28/04/2026

### Objetivo
En Machine Learning no solo entrenamos modelos: también tenemos que servirlos, es decir, ponerlos en producción para que otras personas o sistemas puedan usarlos y obtener predicciones.

### Información útil
+ Medium:
  + [📄Beyond .pkl: The Complete Guide to Saving Machine Learning Models](https://medium.com/@rohanmistry231/beyond-pkl-the-complete-guide-to-saving-machine-learning-models-ddefaeb3e53e)
  + [Demystifying Pickling: How Serialization Works for Data and ML Models](https://medium.com/@tzhaonj/demystifying-pickling-how-serialization-works-for-data-and-ml-models-85b2caedc8ed)
+ [Scikit-Learn](https://scikit-learn.org/stable/model_persistence.html)
+ Towards Data Science:
  + [5 Different Ways To Save Your Machine Learning Model](https://towardsdatascience.com/5-different-ways-to-save-your-machine-learning-model-b7996489d433/)
  + [Still Using the OS Module in Python? This Alternative is Remarkably Better](https://towardsdatascience.com/still-using-the-os-module-in-python-this-alternative-is-remarkably-better-7d728ce22fb7/)
---


## 0. Configuración del Notebook
Importaremos todas las librerías y funciones que vemos relevantes para el notebook que vamos a crear. Lo ideal sería tener todas las importaciones juntas, de tal manera que se puedan controlar de manera sencilla.

### Importación de librerías

In [1]:
import joblib as jl
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import pickle as pk
import plotly.express as px
import re
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        recall_score,
        precision_score,
        roc_auc_score
    )
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Importamos esta librería para solventar el problema que podáis tener para visualizar gráficos con Plotly
from IPython.display import HTML

### Definición de constantes

In [3]:
# Paths
PATH_DIRECTORIO_DATOS = "../data"

PATH_DATASET_ADVERTISING = f"{PATH_DIRECTORIO_DATOS}/advertising.csv"
PATH_DATASET_CALORIES = f"{PATH_DIRECTORIO_DATOS}/calories.csv"
PATH_DATASET_CALORIES_LITE = f"{PATH_DIRECTORIO_DATOS}/calories_time_reduc.csv"
PATH_DATASET_CALORIES_TIME = f"{PATH_DIRECTORIO_DATOS}/calories_time.csv"
PATH_DATASET_HEART = f"{PATH_DIRECTORIO_DATOS}/heart_disease.csv"
PATH_DATASET_BOSTON = f"{PATH_DIRECTORIO_DATOS}/housing_boston.csv"
PATH_DATASET_TWEETS = f"{PATH_DIRECTORIO_DATOS}/nlp_tweets.csv"

# Datasets de Internet
URL_DATASET_PENGUINS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

In [4]:
IF_PLOTLY_FAILS = False

## 1. Carga de datos

In [5]:
# Cargamos el dataset de Heart Disease
df_heart = pd.read_csv(PATH_DATASET_HEART)

In [6]:
# Mostramos las primeras filas del dataset
df_heart.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [7]:
# Mostramos la información del dataset
df_heart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [8]:
# Mostramos las estadísticas del dataset
df_heart.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.366337,9.082101,29.0,47.5,55.0,61.0,77.0
sex,303.0,0.683168,0.466011,0.0,0.0,1.0,1.0,1.0
cp,303.0,0.966997,1.032052,0.0,0.0,1.0,2.0,3.0
trestbps,303.0,131.623762,17.538143,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.264026,51.830751,126.0,211.0,240.0,274.5,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.528053,0.525860,0.0,0.0,1.0,1.0,2.0
thalach,303.0,149.646865,22.905161,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## 2. Preprocesamiento de datos


### Separación de datos
Se dividen los datos en conjuntos de entrenamiento y prueba, para el desarrollo y evaluación del modelo. Además se realiza de manera estratificada para mantener la proporción de la variable objetivo en ambos conjuntos.


In [9]:
# Elige la variable objetivo (cambia si tu dataset usa otro nombre)
target_heart = 'target'

# Separamos el dataset en entrenamiento y test
X_train_heart, X_test_heart, y_train_heart, y_test_heart = train_test_split(
    df_heart.drop(columns=[target_heart]),
    df_heart['target'],
    test_size=0.2,
    random_state=42,
    stratify=df_heart['target']
)

### Escalado de datos
Haremos en dos fases:
- Primera: ajustar el escalador con el dataset de entrenamiento
- Segunda: transformar los datasets de entrenamiento y test

Esto es importante porque el escalador debe ajustarse solo con el dataset de entrenamiento y, una vez serializado el modelo, el escalador también debe ser serializado para que pueda ser utilizado en el despliegue.

In [10]:
# Se instancia el objeto scaler
scaler = StandardScaler()

# Ajustar el scaler SOLO con los datos de entrenamiento
scaler.fit(X_train_heart)

# Transformar ambos conjuntos
X_train_scaled = scaler.transform(X_train_heart)
X_test_scaled = scaler.transform(X_test_heart)

## 3. Entrenamiento del modelo
Se optará por un `RandomForestClassifier` pero se podría haber optado por cualquier otro.

In [11]:
# Entrenamiento de un modelo de RandomForest Classifier
modelo_rfc = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
modelo_rfc.fit(X_train_scaled, y_train_heart)

RandomForestClassifier(max_depth=3, random_state=42)

In [12]:
# Hacemos predicciones sobre el conjunto de test
y_pred = modelo_rfc.predict(X_test_scaled)

# Obtención de las métricas de evaluación
acc = accuracy_score(y_test_heart, y_pred)
prec = precision_score(y_test_heart, y_pred)
rec = recall_score(y_test_heart, y_pred)
f1 = f1_score(y_test_heart, y_pred)

# Cálculo del AUC (Area Under the Curve)
y_proba = modelo_rfc.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test_heart, y_proba)

print(f"Accuracy:  {acc:.2f}")
print(f"Precisión: {prec:.2f}")
print(f"Recall:    {rec:.2f}")
print(f"F1-Score:  {f1:.2f}")
print(f"AUC:       {auc:.2f}\n")


Accuracy:  0.82
Precisión: 0.76
Recall:    0.97
F1-Score:  0.85
AUC:       0.92



## 4. Serialización

| Método                                  | Frameworks habituales                            | Pros                                                                                                                                                                | Contras                                                                                                                                  |
| --------------------------------------- | ------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| **Pickle / Joblib**                     | scikit-learn, modelos Python puros               | - Muy sencillo de usar<br>- Rápido<br>- Ideal para modelos clásicos (RandomForest, LogisticRegression, etc.)<br>- Estándar en Python                                | - Solo funciona en entornos Python<br>- Falta de compatibilidad entre versiones<br>- No seguro al cargar ficheros externos (pickle risk) |
| **ONNX (Open Neural Network Exchange)** | scikit-learn (via skl2onnx), PyTorch, TensorFlow | - Formato estándar multiplataforma<br>- Permite servir modelos en otros lenguajes (C++, C#, JS…)<br>- Optimizaciones para producción                                | - Conversión no siempre perfecta<br>- No todos los modelos están soportados<br>- Requiere tooling adicional                              |
| **SavedModel (TensorFlow)**             | TensorFlow / Keras                               | - Formato nativo de TensorFlow<br>- Compatible con TensorFlow Serving, TFLite y TF.js<br>- Contiene grafo + pesos + firma de predicción                             | - Solo válido para TensorFlow<br>- Más complejo para alumnos que empiezan                                                                |
| **HDF5 (.h5)**                          | TensorFlow / Keras                               | - Muy usado en Keras<br>- Fácil de guardar/cargar<br>- Portabilidad buena                                                                                           | - No guarda la estructura completa del grafo<br>- Menos adecuado para producción comparado con SavedModel                                |
| **MLflow Model Format**                 | Cualquier framework                              | - Estándar MLOps multiplataforma<br>- Permite versionado, trazabilidad y despliegue unificado<br>- Soporta múltiples formatos internos (pickle, ONNX, TorchScript…) | - Requiere entorno MLflow<br>- Curva de aprendizaje mayor                                                                                |


Veremos dos formas:
 - Joblib: típica y sencilla
 - MLflow: más avanzada pero es el estándar a día de hoy

### 4.1 Joblib

#### Guardar el modelo 💾
Se genera un archivo binario (serializar) para poder guardar el objeto que contiene el modelo entrenado. Se puede guardar el modelo en un archivo `.pkl` o `.joblib`.

Utilizaremos la librería `joblib`:
```python
import joblib
```

In [13]:
import joblib
from pathlib import Path

PATH_MODELS = Path("../models/")
PATH_MODELS.mkdir(parents=True, exist_ok=True)

In [14]:
# Usamos with para abrir los archivos con un context manager
# Abrimos el archivo en modo escritura binaria

# Guardamos el modelo
FILENAME_MODELO = PATH_MODELS.joinpath("model_heart_rf.pkl")
with open(FILENAME_MODELO, "wb") as f:
    joblib.dump(modelo_rfc, f)

# Guardamos el objeto de StandardScaler, ya que se necesitará posteriormente para la inferencia
FILENAME_SCALER = PATH_MODELS.joinpath("scaler_heart.pkl")
with open(FILENAME_SCALER, "wb") as f:
    joblib.dump(scaler, f)

#### Cargamos y leemos el modelo 🚀
En esta sección, simularemos el proceso de poner un modelo de Machine Learning en producción. Esto implica cargar un modelo previamente entrenado y guardado, para luego utilizarlo para hacer predicciones sobre nuevos datos, tal como lo haría un sistema en un entorno real de producción. No entrenaremos el modelo de nuevo, sino que nos enfocaremos en su despliegue y uso.

In [15]:
# Cargar el modelo entrenado
with open(FILENAME_MODELO, 'rb') as f:
    modelo_cargado = joblib.load(f)

# Cargar el escalador
with open(FILENAME_SCALER, 'rb') as f:
    scaler_cargado = joblib.load(f)

In [16]:
dict_data = [{
  "age": 57,
  "sex": 1,
  "cp": 0,
  "trestbps": 130,
  "chol": 236,
  "fbs": 0,
  "restecg": 1,
  "thalach": 174,
  "exang": 0,
  "oldpeak": 0.0,
  "slope": 1,
  "ca": 1,
  "thal": 2
}]

In [17]:
X_inferencia_heart = pd.DataFrame().from_dict(dict_data, orient='columns')

In [18]:
X_inferencia_heart_escalado = scaler_cargado.transform(X_inferencia_heart)

In [19]:
y_pred = modelo_cargado.predict(X_inferencia_heart_escalado)
y_proba = modelo_cargado.predict_proba(X_inferencia_heart_escalado)

print(f"La clase predicha es {y_pred[0]} y la probabilidad es {y_proba[0][1]:.2%}")

La clase predicha es 1 y la probabilidad es 54.97%


### 4.2 MLflow: Registro y Serving
A continuación, utilizaremos `MLflow` para registrar el modelo entrenado y posteriormente cargarlo para realizar predicciones.

In [20]:
import mlflow

# Configuración del experimento
mlflow.set_experiment("Heart_Disease_Project")

/Users/mac-SMARTI25/Documents/personal_projects/pontia-master-ia-machine-learning-0925/env-pontia-ml/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2026/04/29 00:04:57 INFO mlflow.tracking.fluent: Experiment with name 'Heart_Disease_Project' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/mac-SMARTI25/Documents/personal_projects/pontia-master-ia-machine-learning-0226/notebooks/mlruns/816796937949326613', creation_time=1777413897422, experiment_id='816796937949326613', last_update_time=1777413897422, lifecycle_stage='active', name='Heart_Disease_Project', tags={}>

In [21]:
from sklearn.pipeline import Pipeline

# Crear pipeline con los objetos ya entrenados
pipeline = Pipeline([
    ('scaler', scaler),
    ('model', modelo_rfc)
])

# Iniciar el run
with mlflow.start_run(run_name="Heart_Disease_RF_Pipeline") as run:
    # Log de parámetros
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 3,
        "random_state": 42
    })
    
    # Log de métricas
    mlflow.log_metrics({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "auc": auc
    })
    
    # Log del pipeline (incluye scaler y modelo)
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="model_heart",
        registered_model_name="HeartDiseasePipeline"
    )
    
    print(f"Run ID: {run.info.run_id}")
    print("Pipeline registrado exitosamente en MLflow.")

2026/04/29 00:05:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/29 00:05:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 62e46c495c2c4906bed8319a9207d830
Pipeline registrado exitosamente en MLflow.


/Users/mac-SMARTI25/Documents/personal_projects/pontia-master-ia-machine-learning-0925/env-pontia-ml/lib/python3.11/site-packages/mlflow/tracking/_model_registry/utils.py:215: FutureWarning: Filesystem model registry backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri)
Successfully registered model 'HeartDiseasePipeline'.
Created version '1' of model 'HeartDiseasePipeline'.


In [22]:
# Cargar el pipeline desde el Model Registry
model_uri = "models:/HeartDiseasePipeline/latest"
loaded_pipeline = mlflow.sklearn.load_model(model_uri)

In [23]:
# Realizar inferencia usando el pipeline (acepta datos crudos)
# Nota: Usamos X_inferencia_heart (DataFrame original) en lugar de X_inferencia_heart_escalado
y_pred_mlflow = loaded_pipeline.predict(X_inferencia_heart)
y_proba_mlflow = loaded_pipeline.predict_proba(X_inferencia_heart)

print(f"Predicción con MLflow: {y_pred_mlflow[0]}")
print(f"Probabilidad con MLflow: {y_proba_mlflow[0][1]:.2%}")

Predicción con MLflow: 1
Probabilidad con MLflow: 54.97%
